---
title: "DRG Checks"

author: "Carlos Resurreccion"

date: "2025-04-03"
---


# Parameters

Change which year_to_load to process in
`~/pids-drg-claims/data-cleaning/debug/cache/year_to_load.txt`

Change main GLOBAL (i.e. across all scripts) parameters in
`~/pids-drg-claims/data-cleaning/00a-parameters.r`

Change seldom touched parameters in
`~/pids-drg-claims/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R`


In [ ]:
source(here::here("data-cleaning", "00a-parameters.r"))


# Libraries


In [ ]:
source(here::here("data-cleaning", "00b-packages.r"))


# R Scripts


In [ ]:
source(here::here("data-cleaning", "00c-load-params-and-scripts.r"))


# Load Mapping Data


In [ ]:
source(here::here("data-cleaning", "00d-load-mapping.r"))


# Data Verification Proper


## Load final .rds


In [ ]:
dt_clean <- readRDS(here(
  chkpt_2_path,
  paste0(
    chkpt_2_prefix, year_to_load, suffix,
    "v2", "_part_b_bq_subset", ".rds"
  )
))


## Load raw file


In [ ]:
dt_raw <- fread(
  file = here(
    raw_claims_path,
    paste0(full_claims_prefix, year_to_load, file_type)
  ), colClasses = "character", header = TRUE,
  encoding = "Latin-1", na.strings = na_values
)

# Drop columns
cols_to_drop <- intersect(colnames(dt_raw), c(drop_cols, drop_cols_manual))
if (length(cols_to_drop) > 0) {
  dt_raw <- dt_raw[, (cols_to_drop) := NULL]
}
str(dt_raw)


## Function Definitions


In [ ]:
test_checks <- function(section_id) {
  # Coerce to two-character string (e.g., 1 → "01")
  section_id <- sprintf("%02d", as.integer(section_id))

  # Get the calling environment (e.g., global or wherever this is invoked from)
  calling_env <- parent.frame()

  # Build pattern to match only variables for the given section
  pattern <- paste0("^chk_", section_id, "_\\d{2}_.+")

  # List relevant check variables in the calling environment
  chk_vars <- ls(envir = calling_env, pattern = pattern)

  # If no matching checks found, warn and exit
  if (length(chk_vars) == 0) {
    message("⚠️ No checks found for section: ", section_id)
    return(invisible(NULL))
  }

  # Get values (assumed format: c(flag, info))
  chk_values_raw <- lapply(chk_vars, get, envir = calling_env)

  # Extract just the logical flag from each
  chk_flags <- sapply(chk_values_raw, function(x) isTRUE(x[1]))

  # Check for failures
  if (any(!chk_flags)) {
    failed_checks <- chk_vars[!chk_flags]

    # Build detailed failure messages
    failure_messages <- mapply(function(var, val) {
      suffix <- sub(paste0("^chk_", section_id, "_\\d{2}_(.+)$"), "\\1", var)
      info <- if (length(val) > 1) val[2] else "No additional info"
      paste0("• ", suffix, ": ", info)
    }, var = failed_checks, val = chk_values_raw[!chk_flags], SIMPLIFY = TRUE)

    stop(paste0(
      "❌ Validation failed in section ", section_id, ":\n",
      paste(failure_messages, collapse = "\n")
    ))
  } else {
    # Print all check results
    cat(paste0("✅ All validation checks passed for section ", section_id, ":\n"))
    for (i in seq_along(chk_vars)) {
      suffix <- sub(paste0("^chk_", section_id, "_\\d{2}_(.+)$"), "\\1", chk_vars[i])
      cat(paste0(suffix, ": ", chk_values_raw[[i]][1], "\n"))
    }
  }
}

custom_setdiff <- function(x, y) {
  # If equal, return TRUE with no message
  if (setequal(x, y)) {
    return(c(TRUE, NULL))
  } else {
    # Identify elements missing and extra
    missing_in_y <- setdiff(x, y) # present in x but missing in y
    extra_in_y <- setdiff(y, x) # present in y but not in x

    # Compose detailed message
    mismatch_msg <- paste0(
      if (length(missing_in_y)) {
        paste0("\n  - missing: ", paste(missing_in_y, collapse = ", "))
      } else {
        ""
      },
      if (length(extra_in_y)) {
        paste0("\n  - invalid: ", paste(extra_in_y, collapse = ", "))
      } else {
        ""
      }
    )

    return(c(FALSE, mismatch_msg))
  }
}

custom_setequal <- function(x, y) {
  # If equal, return TRUE with no message
  if (setequal(x, y)) {
    return(c(TRUE, NULL))
  } else {
    mismatch_msg <- paste0(
      "X: ", x, " Y: ", y
    )
    return(c(FALSE, mismatch_msg))
  }
}


## Test Batch 01:


In [ ]:
cols_expected <- bq_cols
cols_actual <- colnames(dt_clean)
cols_schema <- fromJSON(here(
  "data-cleaning/r_scripts_v2",
  "bq_schema_cleaning.json"
))$name
chk_01_01_cols_match_expected <- custom_setdiff(cols_expected, cols_actual)
chk_01_02_cols_match_schema <- custom_setdiff(cols_schema, cols_actual)
test_checks(1)


## Test Batch 02:


In [ ]:
nrow_expected <- fread(file = here(
  raw_claims_path,
  paste0(full_claims_prefix, year_to_load, file_type)
), select = 1L)[, .N]
nrow_actual_full <- nrow(dt_clean)
chk_02_01_nrows_match_expected_full <- custom_setequal(
  nrow_expected, nrow_actual_full
)
nrow_actual_partial <- nrow_partial <- 0
for (loop_part in 1:split_parts) {
  nrow_partial <- nrow(read_appropriate_file(loop_part))
  nrow_actual_partial <- nrow_actual_partial + nrow_partial
}
chk_02_02_nrows_match_expected_partial <- custom_setequal(
  nrow_expected, nrow_actual_partial
)
test_checks(2)


## Test Batch 03:


In [ ]:
id_series_rows <- dt_clean[grepl("e", id_series), .(id_series)]
id_pin_rows <- dt_clean[grepl("e", id_pin), .(id_series, id_pin)]
id_hci_rows <- dt_clean[grepl("e", id_hci), .(id_series, id_hci)]
id_series_expo_form <- if (!is.null(id_series_rows)) {
  nrow(id_series_rows)
} else {
  0
}
id_pin_expo_form <- if (!is.null(id_pin_rows)) {
  nrow(id_pin_rows)
} else {
  0
}
id_hci_expo_form <- if (!is.null(id_hci_rows)) {
  nrow(id_hci_rows)
} else {
  0
}
chk_03_01_id_series_expo_form <- custom_setequal(
  id_series_expo_form, 0
)
chk_03_02_id_pin_expo_form <- custom_setequal(
  id_pin_expo_form, 0
)
chk_03_03_id_hci_expo_form <- custom_setequal(
  id_hci_expo_form, 0
)
test_checks(3)


In [ ]:
fread(file = here(
  raw_claims_path,
  paste0(full_claims_prefix, year_to_load, file_type)
), select = 1L)
nrow(dt_clean[is.na(clin_discharge)])
